# QAOA — распределённая охота (Colab-шард)

Один «охотничий» прогон по h_train: **kicks + Adam + coord + L-BFGS** (ratchet, как lucky.py).

**Как гонять (5 параллельных колабов):**
1. Файлы → загрузить в /content: `h_train.npy`, `J.npy` и (ОЧЕНЬ рекомендуется) твой текущий лучший CSV — по умолчанию ищется `lucky.csv` (имя задаётся параметром `BASE` в ячейке параметров). Без него — старт со случайных рестартов (дольше, но ок).
2. Runtime → **Run all**.
3. В конце скачаются `shard_{SEED}.csv` + `shard_{SEED}_meta.txt` (каждый запуск = свой случайный SEED, он пишется в лог и в имена).
4. На своём ПК слить: `python merge_shards.py --h h_train.npy --base lucky.csv shard_*.csv --out merged.csv` — покажет, кто кого поймал.

Шард сейвится **после каждого прохода** — смерть сессии не теряет результат (просто перезапусти).

GPU (T4) — идеально. На CPU: измени параметры в ячейке ниже (STEPS=50, KICKS=40, ROUNDS=3).


In [ ]:
import os, time, random
import numpy as np
import torch

# ================= ПАРАМЕТРЫ =================
SEED   = random.randint(1, 10_000_000)  # случайный для каждого запуска
ROUNDS = 8        # проходов
HUNT   = 80       # худшие N (0 = все 500)
KICKS  = 150      # kicks за проход (CPU: 40)
STEPS  = 150      # Adam-шагов на kick (CPU: 50)
COORD  = 1        # раундов coord-доводки (0 = выкл)
LBFGS_IT = 30     # итераций L-BFGS
ALERT  = 0.05     # 'поймали' = прыжок больше этого
BAND_LOW  = 0.0     # полоса охоты: P >= BAND_LOW (0 = выкл)
BAND_HIGH = 1.0     # полоса охоты: P < BAND_HIGH (1.0 = выкл)
WIDE_FRAC  = 0.3  # доля wide-киков: лучшие углы x {1.5,2,3,4} (пробивает hard h)
INFORMED_FRAC = 0.2  # доля informed-киков: узкие γ,β~U(0,0.6) + 1-3 слоя β≈π/2
BASE   = "lucky.csv"  # имя твоего лучшего CSV (база) — то, что загружаешь
OUT_DIR = "/content"
# =============================================
print("SEED =", SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE, "| torch", torch.__version__)
if DEVICE == "cpu":
    print("!! CPU-режим: рекомендую STEPS=50, KICKS=40, ROUNDS=3")


In [ ]:
# QAOA (встроена, файл QAOA.py не нужен)
import torch
import numpy as np

P = 5
N_QUBITS = 12


class QAOA:
    """Дифференцируемый симулятор схемы QAOA глубины p=5 для 12-кубитной модели Изинга.

    J фиксирована, h — вектор линейных членов (вход). По заданным углам gamma, beta
    считает квантовое состояние и метрику P(ground). Углы не подбирает — оценивает.
    Все операции на torch и дифференцируемы по углам: можно обучать модель backprop'ом.
    """

    def __init__(self, J, device="cpu"):
        self.device = device
        self.n = J.shape[0]
        self.p = P
        self.dim = 2 ** self.n

        J = torch.as_tensor(J, dtype=torch.float32, device=device)
        J = (J + J.T) / 2
        J.fill_diagonal_(0)
        self.J = J

        bits = torch.arange(self.dim, device=device)
        x = ((bits.unsqueeze(1) >> torch.arange(self.n - 1, -1, -1, device=device)) & 1).float()
        self.S = 2 * x - 1
        self.quad = 0.5 * torch.einsum("ij,ki,kj->k", self.J, self.S, self.S)

        # быстрый миксер: exp(-i*beta*sum_k X_k) факторизуется по кубитам
        # (RX на разных кубитах коммутируют) -> применяем группами по 4 (bmm).
        # ТОЧНО (расхождение с по-кубитному <= 5e-9), ~5x быстрее по памяти.
        self.g = 4
        self.ng = self.n // self.g
        gb = torch.arange(2 ** self.g, device=device)
        gbits = (gb.unsqueeze(1) >> torch.arange(self.g - 1, -1, -1, device=device)) & 1
        k = (gbits.unsqueeze(1) ^ gbits.unsqueeze(0)).sum(-1)
        self.k = k
        self.phase = torch.tensor([1, -1j, -1, 1j], dtype=torch.complex64,
                                  device=device)[k % 4]

    def energies(self, h):
        h = torch.as_tensor(h, dtype=torch.float32, device=self.device)
        if h.ndim == 1:
            h = h.unsqueeze(0)
        return self.quad.unsqueeze(0) + h @ self.S.T

    def _mixer(self, state, beta):
        B = state.shape[0]
        c = torch.cos(beta).view(B, 1, 1)
        s = torch.sin(beta).view(B, 1, 1)
        amp = c ** (self.g - self.k) * s ** self.k
        M = (amp.to(torch.complex64) * self.phase).transpose(1, 2)
        d = 2 ** self.g
        for j in range(self.ng):
            left, right = d ** j, d ** (self.ng - 1 - j)
            v = state.view(B, left, d, right).permute(0, 1, 3, 2).reshape(B, left * right, d)
            v = torch.bmm(v, M)
            state = v.view(B, left, right, d).permute(0, 1, 3, 2).reshape(B, self.dim)
        return state

    def state(self, h, gamma, beta):
        E = self.energies(h)
        B = E.shape[0]
        gamma = torch.as_tensor(gamma, dtype=torch.float32, device=self.device)
        beta = torch.as_tensor(beta, dtype=torch.float32, device=self.device)
        if gamma.ndim == 1:
            gamma = gamma.unsqueeze(0).expand(B, -1)
        if beta.ndim == 1:
            beta = beta.unsqueeze(0).expand(B, -1)
        psi = torch.full((B, self.dim), 1 / np.sqrt(self.dim), dtype=torch.complex64, device=self.device)
        for l in range(self.p):
            psi = psi * torch.exp(1j * (gamma[:, l].unsqueeze(1) * E))
            psi = self._mixer(psi, beta[:, l])
        return psi

    def probs(self, h, gamma, beta):
        return self.state(h, gamma, beta).abs() ** 2

    def p_ground(self, h, gamma, beta):
        E = self.energies(h)
        prob = self.probs(h, gamma, beta)
        gmin = E.min(dim=1, keepdim=True).values
        mask = (E <= gmin + 1e-9).float()
        return (prob * mask).sum(dim=1)




In [ ]:
miss = [f for f in ["h_train.npy", "J.npy"]
        if not os.path.exists(os.path.join(OUT_DIR, f))]
if miss:
    raise FileNotFoundError("Нет в /content: " + ", ".join(miss) +
        "\n-> Files -> загрузи файлы, затем повтори эту ячейку")
h = np.load(os.path.join(OUT_DIR, "h_train.npy")).astype(np.float32)
J = np.load(os.path.join(OUT_DIR, "J.npy"))
qaoa = QAOA(J, device=DEVICE)
ht = torch.tensor(h, dtype=torch.float32, device=DEVICE)
B = h.shape[0]
print("h:", h.shape, "| J:", J.shape)
mega = os.path.join(OUT_DIR, BASE)
have_mega = os.path.exists(mega)
print(f"{BASE} (база):", "найдена" if have_mega
      else "НЕТ — старт со случайных рестартов (медленнее, но ок)")


In [ ]:
N_ANGLES = 10
cols = ["id"] + [f"gamma_{k}" for k in range(P)] + [f"beta_{k}" for k in range(P)]

def p_eval(a10):
    g = torch.tensor(a10[:, :P], dtype=torch.float32, device=DEVICE)
    b = torch.tensor(a10[:, P:], dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        return qaoa.p_ground(ht, g, b).cpu().numpy()

if have_mega:
    d = np.genfromtxt(mega, delimiter=",", skip_header=1)
    ids = d[:, 0].astype(int)
    order = np.argsort(ids)
    a = d[order, 1:11].astype(np.float32)
    p = p_eval(a)
    print(f"база из {BASE}: mean = %.4f" % p.mean())
else:
    ids = np.arange(B)
    N_RE, N_ST = 20, 120
    rng = np.random.default_rng(SEED)
    print(f"база: {N_RE} случайных рестартов x Adam {N_ST} (батч {B})...")
    a = None
    for rr in range(N_RE):
        g = torch.tensor(rng.uniform(0, 2*np.pi, (B, P)),
                         dtype=torch.float32, device=DEVICE, requires_grad=True)
        b = torch.tensor(rng.uniform(0, 2*np.pi, (B, P)),
                         dtype=torch.float32, device=DEVICE, requires_grad=True)
        opt = torch.optim.Adam([g, b], lr=0.05)
        t0 = time.time()
        for _ in range(N_ST):
            opt.zero_grad()
            loss = -qaoa.p_ground(ht, g, b).mean()
            loss.backward()
            opt.step()
        with torch.no_grad():
            p = qaoa.p_ground(ht, g, b).cpu().numpy()
        a10 = np.concatenate([g.cpu().numpy(), b.cpu().numpy()], 1)
        if a is None:
            a, best_p = a10.copy(), p.copy()
        else:
            upd = p > best_p
            a[upd] = a10[upd]; best_p[upd] = p[upd]
        if rr % 5 == 0 or rr == N_RE - 1:
            print(f"  рестарт {rr+1}/{N_RE}: best-of mean = {best_p.mean():.4f} "
                  f"({time.time()-t0:.0f} c)")
    p = best_p
    print("база готова: mean = %.4f" % p.mean())


In [ ]:
out_csv = os.path.join(OUT_DIR, f"shard_{SEED}.csv")
meta_path = os.path.join(OUT_DIR, f"shard_{SEED}_meta.txt")

def save_shard():
    out = np.concatenate([ids[:, None], a], 1)
    np.savetxt(out_csv, out, delimiter=",", header=",".join(cols),
               comments="", fmt=["%d"] + ["%.12f"]*N_ANGLES)

def adam_chunk(ht_c, a10, steps):
    g = torch.tensor(a10[:, :P], dtype=torch.float32, device=DEVICE, requires_grad=True)
    b = torch.tensor(a10[:, P:], dtype=torch.float32, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([g, b], lr=0.05)
    for _ in range(steps):
        opt.zero_grad()
        loss = -qaoa.p_ground(ht_c, g, b).mean()
        loss.backward()
        opt.step()
    with torch.no_grad():
        p = qaoa.p_ground(ht_c, g, b).cpu().numpy()
    return np.concatenate([g.detach().cpu().numpy(), b.detach().cpu().numpy()], 1), p

def lbfgs_one(i, a_row, iters):
    ht_i = ht[i:i+1]
    g = torch.tensor(a_row[:P][None, :], dtype=torch.float32, device=DEVICE, requires_grad=True)
    b = torch.tensor(a_row[P:][None, :], dtype=torch.float32, device=DEVICE, requires_grad=True)
    def closure():
        opt.zero_grad()
        loss = -qaoa.p_ground(ht_i, g, b).mean()
        loss.backward()
        return loss
    opt = torch.optim.LBFGS([g, b], lr=1.0, max_iter=iters, history_size=10)
    opt.step(closure)
    with torch.no_grad():
        p = qaoa.p_ground(ht_i, g, b).item()
    return np.concatenate([g.detach().cpu().numpy().ravel(), b.detach().cpu().numpy().ravel()]), p

def coord_round(a_sel, p_sel, ht_sel, grid=32):
    H = a_sel.shape[0]
    a_best, p_best = a_sel.copy(), p_sel.copy()
    gv = np.linspace(0, 2*np.pi, grid, endpoint=False)
    for k in range(N_ANGLES):
        for c0 in range(0, grid, 8):
            G = gv[c0:c0+8]; nn = len(G)
            a10c = np.repeat(a_best[:, None, :], nn, axis=1)
            a10c[:, :, k] = G
            a10c = a10c.reshape(-1, N_ANGLES)
            ht_c = ht_sel.repeat_interleave(nn, dim=0)
            g = torch.tensor(a10c[:, :P], dtype=torch.float32, device=DEVICE)
            b = torch.tensor(a10c[:, P:], dtype=torch.float32, device=DEVICE)
            with torch.no_grad():
                pc = qaoa.p_ground(ht_c, g, b).cpu().numpy().reshape(H, nn)
            am = pc.argmax(1); pm = pc.max(1)
            upd = pm > p_best
            a_best[upd] = a10c.reshape(H, nn, N_ANGLES)[np.where(upd)[0], am[upd]]
            p_best[upd] = pm[upd]
        print(f"    coord угол {k+1}/10: mean = {p_best.mean():.4f}", flush=True)
    return a_best, p_best

p_start_full = p.copy()
log_lines = [f"SEED={SEED} DEVICE={DEVICE} start_mean={p.mean():.4f} "
             f"ROUNDS={ROUNDS} HUNT={HUNT} KICKS={KICKS} STEPS={STEPS} "
             f"COORD={COORD} LBFGS={LBFGS_IT} "
             f"t0={time.strftime('%Y-%m-%d %H:%M:%S')}"]


In [ ]:
for r in range(ROUNDS):
    if HUNT > 0:
        order = np.argsort(p)  # худшие первыми
        if 0 < BAND_LOW or BAND_HIGH < 1.0:  # полоса активна
            m = (p[order] >= BAND_LOW) & (p[order] < BAND_HIGH)
            in_b, out_b = order[m], order[~m]
            sel = np.array(list(in_b) + list(out_b))[:HUNT]
            print(f"  полоса {BAND_LOW:g}..{BAND_HIGH:g}: в ней {len(in_b)}, "
                  f"беру худшие {min(len(in_b), HUNT)}"
                  + (f" + добор {HUNT - len(in_b)} извне" if len(in_b) < HUNT else ""), flush=True)
        else:
            sel = order[:HUNT]
    else:
        sel = np.arange(B)
    p_pass = p.copy()
    print(f"\n=== ПРОХОД {r+1}/{ROUNDS}: худшие {len(sel)} "
          f"(P {p[sel].min():.3f}..{p[sel].max():.3f}) ===", flush=True)
    t0 = time.time()
    # --- kicks + Adam ---
    rng = np.random.default_rng(SEED + 1000 * (r + 1))
    sig = np.logspace(np.log10(0.3), np.log10(3.0), KICKS)
    mults = np.array([1.5, 2.0, 3.0, 4.0])
    def informed(Jn):
        af = rng.uniform(0, 0.6, (Jn, len(sel), N_ANGLES))
        for k in range(Jn):
            for _ in range(rng.integers(1, 4)):
                L = rng.integers(0, P)
                af[k, :, P + L] = np.pi / 2 + rng.normal(0, 0.15, len(sel))
        return af
    a_best, p_best = a[sel].copy(), p[sel].copy()
    for c0 in range(0, KICKS, 8):
        Jn = min(8, KICKS - c0)
        a10 = a[sel][None] + sig[c0:c0+Jn][:, None, None] * \
              rng.normal(0, 1, (Jn, len(sel), N_ANGLES))
        # три семейства киков: local (base+σ·N) / wide (base×mult) / informed (свежие)
        r_ = rng.random(Jn)
        wide = r_ < WIDE_FRAC
        inf_ = (r_ >= WIDE_FRAC) & (r_ < WIDE_FRAC + INFORMED_FRAC)
        if wide.any():
            mw = mults[(c0 + np.arange(Jn)) % 4][wide][:, None, None]
            a10[wide] = a[sel][None] * mw + rng.normal(0, 0.05, a10[wide].shape)
        if inf_.any():
            a10[inf_] = informed(int(inf_.sum()))
        ht_c = torch.cat([ht[sel]] * Jn, dim=0)
        a10n, pn = adam_chunk(ht_c, a10.reshape(-1, N_ANGLES), STEPS)
        pn = pn.reshape(Jn, len(sel)); a10n = a10n.reshape(Jn, len(sel), N_ANGLES)
        am = pn.argmax(axis=0); pm = pn.max(axis=0)
        upd = pm > p_best
        a_best[upd] = a10n[am[upd], np.where(upd)[0]]
        p_best[upd] = pm[upd]
        print(f"  kicks {c0+1}-{c0+Jn}/{KICKS}: chunk mean = {pn.mean():.4f}", flush=True)
    a[sel], p[sel] = a_best, p_best
    # --- coord ---
    if COORD > 0:
        for _ in range(COORD):
            a[sel], p[sel] = coord_round(a[sel], p[sel], ht[sel])
    # --- L-BFGS (последовательно: Colab 2 CPU) ---
    if LBFGS_IT > 0:
        n_imp = 0
        for j, i in enumerate(sel):
            out, pn = lbfgs_one(i, a[i], LBFGS_IT)
            if pn > p[i] + 1e-12:
                a[i], p[i] = out, pn; n_imp += 1
            if j % 20 == 0:
                print(f"  L-BFGS {j+1}/{len(sel)}", flush=True)
        print(f"  L-BFGS: улучшено {n_imp}/{len(sel)}", flush=True)
    # --- видимость: ПОЙМАЛИ / ПОРОГ ---
    gain = p - p_pass
    for i in np.argsort(-gain)[:5]:
        if gain[i] > ALERT:
            print(f"  ** ПОЙМАЛИ #{int(ids[i])}: {p_pass[i]:.3f} -> "
                  f"{p[i]:.3f} (+{gain[i]:.3f})", flush=True)
    for i in np.where((p >= 0.5) & (p_pass < 0.5))[0][:10]:
        print(f"  ** ПОРОГ 0.5: #{int(ids[i])} -> {p[i]:.3f}", flush=True)
    save_shard()
    tail = np.argsort(p)[:10]
    print(f"  ПРОХОД {r+1}: mean = {p.mean():.4f} ({time.time()-t0:.0f} c) | tail: "
          + " ".join(f"#{i}:{p[i]:.3f}" for i in tail), flush=True)
    log_lines.append(f"pass{r+1}: mean={p.mean():.4f}")


In [ ]:
with open(meta_path, "w") as f:
    f.write("\n".join(log_lines)
            + f"\nfinal_mean={p.mean():.4f} done={time.strftime('%Y-%m-%d %H:%M:%S')}\n")
print("============= РЕЗУЛЬТАТ =============")
print("\n".join(log_lines))
print(f"final mean P(ground) = {p.mean():.4f}")
tot = p - p_start_full
print("топ-10 скачков за сессию:")
for i in np.argsort(-tot)[:10]:
    print(f"  #{int(ids[i])}: {p_start_full[i]:.3f} -> {p[i]:.3f} (+{tot[i]:.3f})")
try:
    from google.colab import files
    files.download(out_csv)
    files.download(meta_path)
    print("\nСкачивание началось: shard CSV + meta")
except Exception as e:
    print("Скачивание недоступно:", e, "-> Files -> скачать вручную")
